<a href="https://colab.research.google.com/github/thequantumbruh/LLMs/blob/main/TPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset
from datasets import interleave_datasets
from transformers import AutoTokenizer
import numpy as np
import torch
import math
import torch.nn as nn

#from google.colab import drive
#drive.mount('/content/drive')


#T_tokenizer = AutoTokenizer.from_pretrained(
 #   "/content/drive/MyDrive/Temi_pretrained_model_tknzr",
 #   add_bos_token = True,
 #   add_eos_token = True
#)

In [ ]:
device = torch.

In [ ]:

data_set = load_dataset("HuggingFaceFW/fineweb-edu", split= "train", streaming = True)
factual_dataset = load_dataset("wikimedia/wikipedia", "20231101.en",split = "train", streaming = True)

full_ds = interleave_datasets([data_set, factual_dataset], probabilities = [0.65, 0.35])



def get_training_corpus (data, max_token= 20_000_000):
  vocab_size = 0
  for sample in data:
    text = sample["text"] if isinstance(sample, dict) else sample
    yield text
    vocab_size += len(text.split())
    if vocab_size >= max_token:
      print("corpus Limit Reached")
      break

old_tokenizer = AutoTokenizer.from_pretrained("gpt2")
new_tokenizer = old_tokenizer.train_new_from_iterator(get_training_corpus(full_ds), vocab_size= 16384) #funny me. Passed the streamed data instead of the corpus
new_tokenizer.save_pretrained("/content/drive/MyDrive/Temi_pretrained_model_tknzr")




README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

... (2 outputs left out)

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

corpus Limit Reached


KeyboardInterrupt: 

In [ ]:
"""def get_data (data, tokenizer, token_size: int, file_name: str, block_size: int = 1024):
  current_size = 0
  buffer = []


  with open(file_name, "wb") as file:
    for sample in data:
      text = sample['text'] if isinstance(sample, dict) else sample
      batch_token = tokenizer.encode(text)
      buffer.extend(batch_token)
      while (len(buffer) >= block_size):
        batch = buffer[:block_size]
        buffer = buffer[block_size:]

        batch = np.array(batch, dtype = np.uint16)

        file.write(batch.tobytes())
        current_size += len(batch)

      if current_size >= token_size:
        break
    """

"""def get_data_fast(data, tokenizer,token_size : int, file_name: str, batch_size :int =  1000, context_len :int = 1024):

  total_tokens = 0
  buffer = []
  with open(file_name, "wb") as file:
    for i in range(0, len(data), batch_size):
      sample = data[i:batch_size +i]
      text = sample['text'] if isinstance(sample, dict) else [s['text'] for s in sample]
      batch_token = tokenizer(text, add_special_tokens = True)['input_ids']


      for tokens in batch_token:
        buffer.extend(tokens)

      while len(buffer)>= context_len :
        batch = buffer[:context_len]
        buffer = buffer[context_len:]

        batch = np.array(batch, dtype = np.uint16)
        batch.tobyte()
        file.write(batch)
        total_tokens += len(batch)
        if total_tokens >= token_size:
          break

      if total_tokens >= token_size:
        break"""

def tokenizer_function (batch):
    return T_tokenizer(batch['text'], add_special_tokens= True)

clean_fineweb = data_set.remove_columns([col for col in data_set.features if col != 'text'])
clean_wiki = factual_dataset.remove_columns([col for col in factual_dataset.features if col != 'text'])

# Re-define full_ds with identical schemas
full_ds = interleave_datasets([clean_fineweb, clean_wiki], probabilities=[0.65, 0.35])




train_ds = full_ds.shuffle(buffer_size= 100000, seed = 11)
test_ds = full_ds.shuffle(buffer_size= 100000, seed = 121)
val_ds = full_ds.shuffle(buffer_size= 100000, seed = 42)
train_ds = train_ds.map(tokenizer_function, batch_size= 1000, batched=True)
test_ds = train_ds.map(tokenizer_function, batch_size= 1000, batched=True)
val_ds = val_ds.map(tokenizer_function, batch_size= 1000, batched=True)





In [ ]:
from tqdm.auto import tqdm
import numpy as np

def get_data_fast(data_stream, max_token: int, file_name: str):
  buffer = []
  token_count = 0
  bar = tqdm(total=max_token, unit="tok", unit_scale=True, desc=file_name.split("/")[-1])
  with open(file_name, "wb") as file:
    for batch in data_stream:
      input_ids = batch['input_ids']
      if isinstance(input_ids[0], list):
        for text in input_ids:
          buffer.extend(text)
      else:
        buffer.extend(input_ids)

      n = (len(buffer) // 1024) * 1024        # all full 1024-token chunks at once
      if n:
        file.write(np.array(buffer[:n], dtype=np.uint16).tobytes())
        file.flush()
        buffer = buffer[n:]
        token_count += n
        bar.update(n)
        bar.set_postfix(left_in_buffer=len(buffer))

      if token_count >= max_token:
        break
  bar.close()
  print(f"data_saved {token_count} tokens at {file_name}")
  return token_count

#val_tokens   = get_data_fast(val_ds,   10_000_000,  "/content/drive/MyDrive/val.bin")
#train_tokens = get_data_fast(train_ds, 100_000_000, "/content/drive/MyDrive/train.bin")


In [ ]:
train_tokens = get_data_fast(train_ds, 2_000_000_000, "/content/drive/MyDrive/train.bin")

train.bin:   0%|          | 0.00/2.00G [00:00<?, ?tok/s]

KeyboardInterrupt: 

In [ ]:
class Caus_attn_Mech (nn.Module):
  def __init__(self,in_dim :int, out_dim :int, num_head :int, context_len: int):
    super.__init__()
    assert(in_dim % num_head == 0)
    self.head_dim = in_dim// num_head
    self.num_head = num_head
    self.q_w = nn.Linear(in_dim, out_dim, bias = False)
    self.k_w = nn.Linear(in_dim, out_dim, bias = False)
    self.v_w  = nn.Linear(in_dim, out_dim, bias = False)
    self.register_buffer("mask", torch.triu(torch.ones(context_len, context_len), diagonal =1 ))
    self.drop_out = nn.Dropout(0.12)
    self.output_proj = nn.Linear(out_dim, in_dim)


    def forward(self, x :torch.Tensor) -> torch.Tensor:
      query = self.q_w(x)
      key = self.k_w(x)
      value = self.v_w(x)

      in_shape = x.shape
      query = query.transpose(1,2)#transpose axis (num_head and number of tokens)
      key = key.transpose(1,2)# to align emb_dimension with it its respective token
      value = value.transpose(1,2)


      query = query.contiguous().view(in_shape[0],in_shape[1], self.num_head, self.head_dim)
      key = key.contiguous().view(in_shape[0],in_shape[1], self.num_head, self.head_dim)
      value = value.contiguous().view(in_shape[0],in_shape[1], self.num_head, self.head_dim)

      mask = self.mask.bool()[:in_shape[1], :in_shape[1]]

      attn_weights = query @ key.transpose(2,3)
      attn_weights.masked_fill_(mask, -torch.inf)
      attn_weights = nn.Softmax(attn_weights/math.sqrt(self.head_dim))
      attn_weights = self.drop_out(attn_weights)

      context_vec = (attn_weights @ value).transponse(1,2)
      context_vec = context_vec.contiguous().view(in_shape[0], in_shape[1], in_shape[2])

      return self.output_proj(context_vec)



In [ ]:
import torch
import torch.nn as nn

class FF_Layer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        emb_dim = cfg["emb_dim"]
        self.sequential = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.GELU(),
            nn.Linear(4 * emb_dim, emb_dim)
        )

    def forward(self, x :torch.Tensor) -> torch.Tensor:
        return self.sequential(x)


class Transformer_arc(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.FF = FF_Layer(cfg)
        self.attn = Caus_attn_Mech(
            in_dim=cfg["emb_dim"],
            out_dim=cfg["emb_dim"],
            num_head=cfg["head_num"],
            context_len=cfg["context_len"]
        )
        self.layer_Norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.layer_Norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.dropout = nn.Dropout(cfg["drop_rate"])

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        shortcut = x
        x = self.layer_Norm1(x)
        x = self.attn(x)
        x = self.dropout(x)
        x = shortcut + x  # First residual connection


        shortcut = x
        x = self.layer_Norm2(x)
        x = self.FF(x)
        x = self.dropout(x)
        x = shortcut + x  # Second residual connection

        return x

In [ ]:
import torch
import torch.nn as nn

class TPT_v1(nn.Module):
    def __init__(self, cfg :dict):
        super().__init__()
        self.t_layer = cfg["t_layer"]
        self.transformer_blocks = nn.ModuleList([
            Transformer_arc(cfg) for _ in range(self.t_layer)
        ])
        self.layer_norm = nn.LayerNorm(cfg["emb_dim"])
        self.dropout = nn.Dropout(cfg["drop_rate"])

        self.output_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dropout(x)

        for block in self.transformer_blocks:
            x = block(x)

        x = self.layer_norm(x)
        logits = self.output_head(x)

        return logits

In [ ]:
tpt_config = {
    "drop_rate": 0.12,
    "emd_dim": 1024,
    "vocab_size": 16384,
    "t_layer": 8,
    "context_len": 512,
    "head_num": 8
}

In [ ]:
tpt = TPT_v1(tpt_config)
tpt()

In [ ]:
#training loop

